# Case Study：採購單 B2048 送審前檢查

## 建立採購代理人 Procurement Agent

In [1]:
# load API Key
from dotenv import load_dotenv
from langchain_core.messages import ToolMessage

load_dotenv()

True

In [27]:
# Global variable
MODEL_NAME = "gpt-5-nano"

In [2]:
# System Prompt
# 將商業檢查邏輯封裝在 Tool 中
PROCURE_AGENT_SYSTEM_PROMPT = """
你是 Procurement Agent，負責檢查採購文件是否符合送審前的採購規則。

收到採購單編號後，請使用提供的 Tools 檢查。

你不負責物料需求、預算或最終送審判斷，也不得修改或送出採購單。
"""

In [3]:
# 定義採購檢查結果的輸出格式，使用 Pydantic 的 BaseModel

from pydantic import BaseModel, Field
from typing import Literal

class ProcurementCheckResult(BaseModel):
    document_id: str = Field(..., description="採購單編號")
    status: Literal["ready", "blocked", "unknown"] = Field(..., description="檢查狀態")
    department_id: str = Field(..., description="申請部門編號")
    department: str = Field(..., description="申請部門")
    total_amount: float = Field(..., description="採購總金額")
    check_results: str = Field(..., description="檢查結果與原因")
    recommendations: str = Field(..., description="建議處理方式")

In [4]:
# The function to get procurement document data from a simulated database

def get_procurement_document(document_id: str) -> dict[str, dict]:
    """
    Get procurement document data by document ID.
    """
    # 模擬 採購單 B2048 的資料

    procurement_data = {
        "B2048": {
            "document_type": "採購單",
            "department_id": "dept-01",
            "department": "製造部",
            "material_id": "T1001",
            "material": "溫度感測器",
            "quantity": 8,
            "unit_price": 10000,
            "total_amount": 80000,
            "supplier": "AAA 電子",
            "quote_status": "有效",
            "expected_delivery_date": "2026-08-20"
        }
    }
    return procurement_data

In [5]:
# 檢查以下商業規則 的工具
# 1. 採購總金額為採購數量乘以單價。
# 2. 供應商報價必須有效。

from langchain.tools import tool

@tool
def check_procurement_rules(document_id: str) -> dict[str, str]:
    """
    Check procurement rules for the given document ID.
    """
    # Call helper function to get procurement document data
    document_data = get_procurement_document(document_id)

    # Check if the document exists
    if document_id not in document_data:
        return {
            "status": "unknown",
            "check_results": "採購單不存在。",
            "recommendations": "請確認採購單編號是否正確。"
        }

    # Check 採購總金額為採購數量乘以單價
    doc = document_data[document_id]
    expected_total_amount = doc.get("quantity", 0) * doc.get("unit_price", 0)
    if doc.get("total_amount") != expected_total_amount:
        return {
            "status": "blocked",
            "check_results": f"採購總金額不正確，應為 {expected_total_amount} 元。",
            "recommendations": "請確認採購數量與單價是否正確。"
        }

    # Check 供應商報價是否有效
    if doc.get("quote_status") != "有效":
        return {
            "status": "blocked",
            "check_results": "供應商報價無效。",
            "recommendations": "請確認供應商報價是否有效。"
        }

    # If all checks pass
    return {
        "status": "ready",
        "department_id": doc.get("department_id", ""),
        "department": doc.get("department", ""),
        "total_amount": doc.get("total_amount", 0),
        "check_results": "採購單符合送審前的檢查規則。",
        "recommendations": "可以進行送審。" 
    }
    

In [6]:
# 建立 Procurement Agent，使用 gpt-5-nano 模型，並指定系統提示與工具

from langchain.agents import create_agent

procurement_agent = create_agent(
    model=MODEL_NAME,
    system_prompt=PROCURE_AGENT_SYSTEM_PROMPT,
    tools=[check_procurement_rules],
    response_format=ProcurementCheckResult
    )

### Test the Procurement Agent

In [7]:
# Test the Procurement Agent with a sample procurement document ID
test_document_id = "B2048"
result_of_check_procurement = procurement_agent.invoke(
    input={"messages": [
        {"role": "user", "content": f"請檢查採購單 {test_document_id} 是否符合送審前的規則。"}
    ]}
)

In [14]:
from pprint import pp

pp(result_of_check_procurement)

{'messages': [HumanMessage(content='請檢查採購單 B2048 是否符合送審前的規則。', additional_kwargs={}, response_metadata={}, id='f318e8f6-bd86-42ee-89a6-4699bbd130d6'),
              AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 220, 'prompt_tokens': 499, 'total_tokens': 719, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYnOjaHPi0XjSy9hIX3KbDoGbdq10', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1393e-3f56-78e1-a2fb-5e5e2cb25581-0', tool_calls=[{'name': 'check_procurement_rules', 'args': {'document_id': 'B2048'}, 'id': 'call_yZP4yD24SGhXLA8WhAghuUcr', 'type': 'tool_call'}], inv

In [21]:
# print the result
from pprint import pp
from langchain_core.messages import ToolMessage
from lib.print_helper import print_response

# pp(result_of_check_procurement)
print_response(result_of_check_procurement)


HumanMessage
請檢查採購單 B2048 是否符合送審前的規則。

--------------------------------------------------

AIMessage
tool call:  [{'name': 'check_procurement_rules', 'args': {'document_id': 'B2048'}, 'id': 'call_yZP4yD24SGhXLA8WhAghuUcr', 'type': 'tool_call'}]

--------------------------------------------------

ToolMessage
{"status": "ready", "department_id": "dept-01", "department": "製造部", "total_amount": 80000, "check_results": "採購單符合送審前的檢查規則。", "recommendations": "可以進行送審。"}
tool name:  check_procurement_rules
tool id:  call_yZP4yD24SGhXLA8WhAghuUcr

--------------------------------------------------

AIMessage
{"document_id":"B2048","status":"ready","department_id":"dept-01","department":"製造部","total_amount":80000,"check_results":"採購單符合送審前的檢查規則。","recommendations":"可以進行送審。"}
tool call:  []

--------------------------------------------------

Structured Response:
ProcurementCheckResult(document_id='B2048', status='ready', department_id='dept-01', department='製造部', total_amount=80000.0, check_result

## 建立物料代理人 Material Agent

In [22]:
# System prompt for Material Agent
MATERIAL_AGENT_SYSTEM_PROMPT = """
你是 Material Agent，負責檢查採購內容是否符合物料需求。

收到採購單編號後，使用提供的 Tools 檢查：

- 採購數量是否符合實際需求。
- 供應商預計交貨日是否符合物料需求日。

只根據 Tool 回傳的模擬資料判斷，不得自行補造或修改資料：

- 所有檢查都通過：`ready`
- 任一規則未通過：`blocked`
- 資料不足、結果衝突或 Tool 執行失敗：`unknown`

如果狀態為 `blocked`，請說明未通過原因及建議處理方式。

你不負責檢查採購文件、供應商報價或部門預算，也不得修改或送出採購單。
""" 


In [23]:
#  Material Agent 的輸出格式，使用 Pydantic 的 BaseModel

from pydantic import BaseModel, Field
from typing import Literal

# 假設只有一張採購單內只有一個 material id
class MaterialCheckResult(BaseModel):
    document_id: str = Field(..., description="採購單編號")
    material_id: str = Field(..., description="物料編號")
    status: Literal["ready", "blocked", "unknown"] = Field(..., description="檢查狀態")
    check_results: list[str] = Field(..., description="檢查結果與原因")
    recommendations: list[str] = Field(..., description="建議處理方式")

In [24]:
# The function to get material requirement data from a simulated database

def get_material_data(material_id: str) -> dict[str, dict]:
    """
    Get material requirement data by material ID.
    """
    # 模擬 document B2048 的物料需求資料，包含需求量、現有庫存量與需求日期

    material_data = {
        "T1001": {
            "required_quantity": 10,
            "on_hand_quantity": 2,
            "required_before_date": "2026-08-25"}
    }
    return material_data

In [25]:
# 檢查以下商業規則的工具:
# 1. 採購數量應等於需求量減去現有庫存量。
# 2. 供應商預計交貨日不得晚於物料需求日。
# 3. 數量不足或交貨太晚時，不具備送審條件。

from typing import Dict

from langchain.tools import tool

@tool
def check_material_requirements(document_id: str) -> dict[str, str | list[str]]:
    """
    Check material requirements for the given document ID.
    """

    # Call helper function to get procurement document data
    procurement_data = get_procurement_document(document_id)
    if document_id not in procurement_data:
            return {
                "status": "unknown",
                "check_results": "找不到採購單資料。",
                "recommendations": "請確認採購單編號是否正確，或聯絡採購部門。"
            }

    # Call helper function to get material data by material ID
    material_id = procurement_data[document_id].get("material_id", "")
    material_data = get_material_data(material_id)

    # Check if the document exists
    if material_id not in material_data:
        return {
            "status": "unknown",
            "check_results": "找不到採購單內對應的物料需求資料。",
            "recommendations": "請確認採購單編號是否正確，或聯絡物料管理部門。"
        }

    # Get the procurement document data
    procurement_info = procurement_data[document_id]
    purchase_quantity = procurement_info.get("quantity", 0)
    expected_delivery_date = procurement_info.get("expected_delivery_date", "")
    

    # Get the material requirement data
    material_info = material_data[material_id]
    required_quantity = material_info.get("required_quantity", 0)
    on_hand_quantity = material_info.get("on_hand_quantity", 0)
    short_quantity = required_quantity - on_hand_quantity
    required_before_date = material_info.get("required_before_date", "")

    check_results = []
    recommendations = []
    # Check the quantity rule
    if purchase_quantity < short_quantity:
        check_results.append(f"採購數量 {purchase_quantity} 少於需求量 {short_quantity}。")
        recommendations.append("請增加採購數量以符合需求。")

    # Check the delivery date rule
    if expected_delivery_date > required_before_date:
        check_results.append(f"預計交貨日 {expected_delivery_date} 晚於需求日 {required_before_date}。")
        recommendations.append("請要求供應商提前交貨或調整採購單。")

    if not check_results:
        check_results.append("採購單符合物料需求。")
        recommendations.append("可以進行送審。")
        return {
            "status": "ready",
            "material_id": material_id,
            "document_id": document_id,
            "check_results": check_results,
            "recommendations": recommendations
        }
    else:
        return {
            "status": "blocked",
            "document_id": document_id,
            "material_id": material_info.get("material_id", ""),
            "check_results": "\n".join(check_results),
            "recommendations": "\n".join(recommendations)
        }


In [28]:
# 建立 Material Agent，使用 gpt-5-nano 模型，並指定系統提示與工具

from langchain.agents import create_agent

material_agent = create_agent(
    model=MODEL_NAME,
    system_prompt=MATERIAL_AGENT_SYSTEM_PROMPT,
    tools=[check_material_requirements],
    response_format=MaterialCheckResult
)

In [29]:
# Test the material agent with a sample procurement document ID
test_document_id = "B2048"
result_of_check_material = material_agent.invoke(
    input={"messages": [
        {"role": "user", "content": f"請檢查採購單 {test_document_id} 是否符合物料需求。"}
    ]}
)

In [30]:
# print the result

from lib.print_helper import print_response
print_response(result_of_check_material)

HumanMessage
請檢查採購單 B2048 是否符合物料需求。

--------------------------------------------------

AIMessage
tool call:  [{'name': 'check_material_requirements', 'args': {'document_id': 'B2048'}, 'id': 'call_YgWP2ZtWg33P1gOVpYFy9nML', 'type': 'tool_call'}]

--------------------------------------------------

ToolMessage
{"status": "ready", "material_id": "T1001", "document_id": "B2048", "check_results": ["採購單符合物料需求。"], "recommendations": ["可以進行送審。"]}
tool name:  check_material_requirements
tool id:  call_YgWP2ZtWg33P1gOVpYFy9nML

--------------------------------------------------

AIMessage
{"document_id":"B2048","material_id":"T1001","status":"ready","check_results":["採購單符合物料需求。"],"recommendations":["可以進行送審。"]}
tool call:  []

--------------------------------------------------

Structured Response:
MaterialCheckResult(document_id='B2048', material_id='T1001', status='ready', check_results=['採購單符合物料需求。'], recommendations=['可以進行送審。'])


## 建立財務代理人 Finance Agent

In [31]:
FINANCIAL_AGENT_SYSTEM_PROMPT = """
你是 Finance Agent，負責檢查部門預算是否足以支付採購金額。

在採購文件檢查並通過後，使用提供的 Tools 檢查部門可用預算：

- 可用預算大於或等於採購總金額：`ready`
- 可用預算小於採購總金額：`blocked`
- Tool 執行失敗或結果不明確：`unknown`

只使用已確認的申請部門、採購總金額及 Tool 回傳的資料，不得自行猜測或修改資料。

如果預算不足，請說明不足金額及建議處理方式。

你只負責財務預算檢查，不負責採購文件、物料需求或最終送審判斷，也不得修改預算或送出採購單。
"""

In [32]:
# The output schema for Finance Agent
from pydantic import BaseModel, Field
from typing import Literal

class FinanceCheckResult(BaseModel):
    document_id: str = Field(..., description="採購單編號")
    department_id: str = Field(..., description="申請部門編號")
    total_amount: float = Field(..., description="採購總金額")
    available_budget: float | None = Field(..., description="可用預算")
    status: Literal["ready", "blocked", "unknown"] = Field(..., description="檢查狀態")
    check_results: str = Field(..., description="檢查結果與原因")
    recommendations: str = Field(..., description="建議處理方式")

In [33]:
# function to get department budget data from a simulated database

# 由 Procurement Agent 取得 document_id and total_amount, 再由
def get_department_budget(department_id: str) -> dict[str, dict]:
    """
    Get department budget data by department ID.
    """
    # 模擬部門預算資料，假設只有一個部門 dept-01 的資料
    dept_budget_data = {
        "dept-01": {
            "department": "製造部",
            "available_budget": 100000
        }
    }
    return dept_budget_data


In [34]:
# The tool to check department budget against procurement total amount

from langchain.tools import tool

@tool
def check_department_budget(document_id: str, total_amount:float, department_id: str) -> dict[str, str | float]:
    """
    Check if the department budget is sufficient for the procurement total amount.

    :param document_id: The procurement document ID.
    :param total_amount: The total amount of the procurement.
    :param department_id: The ID of the department requesting the procurement, such as "dept-01".
    """

    dept_id = department_id
    po_total_amount = total_amount

    # call helper function to get department budget data
    budget_data = get_department_budget(dept_id)

    # Check if the department exists
    if dept_id not in budget_data:
        return {
            "status": "unknown",
            "check_results": "找不到申請部門的預算資料。",
            "recommendations": "請確認申請部門是否正確，或聯絡財務部門。"
        }

    # Get the available budget
    available_budget = budget_data[dept_id].get("available_budget", 0)

    # Check if the available budget is sufficient
    status = ("ready" if available_budget >= po_total_amount else "blocked")
    check_result = ("可用預算足以支付採購總金額。" 
                    if available_budget >= po_total_amount 
                    else f"可用預算{available_budget}不足以支付採購總金額{po_total_amount}。")
    recommendation = ("可以進行送審。" 
                      if available_budget >= po_total_amount 
                      else f"請增加部門預算或調整採購金額。")

    # Return the result, need to match the output schema of FinanceCheckResult
    return {
        "document_id": document_id,
        "department_id": dept_id,
        "total_amount": po_total_amount,
        "available_budget": available_budget,
        "status": status,
        "check_results": check_result,
        "recommendations": recommendation
    }

In [35]:
# Create the Finance Agent using the system prompt and the tool

from langchain.agents import create_agent
finance_agent = create_agent(
    model=MODEL_NAME,
    system_prompt=FINANCIAL_AGENT_SYSTEM_PROMPT,
    tools=[check_department_budget],
    response_format=FinanceCheckResult
)

In [36]:
# Test the Finance Agent  with a sample procurement document ID
test_document_id = "B2048"
result_of_check_finance = finance_agent.invoke(
    input={"messages": [
        {"role": "user", "content": f"目前採購單 {test_document_id} 已通過採購規則檢查，請再檢查部門預算是否足夠。採購單的總金額為 80000 元，申請部門為 dept-01。"}
    ]}
)

In [37]:
# Print the result
from lib.print_helper import print_response
print_response(result_of_check_finance)

HumanMessage
目前採購單 B2048 已通過採購規則檢查，請再檢查部門預算是否足夠。採購單的總金額為 80000 元，申請部門為 dept-01。

--------------------------------------------------

AIMessage
tool call:  [{'name': 'check_department_budget', 'args': {'document_id': 'B2048', 'total_amount': 80000, 'department_id': 'dept-01'}, 'id': 'call_vlCS5Gmesl8t24YQPDW7Xxpy', 'type': 'tool_call'}]

--------------------------------------------------

ToolMessage
{"document_id": "B2048", "department_id": "dept-01", "total_amount": 80000.0, "available_budget": 100000, "status": "ready", "check_results": "可用預算足以支付採購總金額。", "recommendations": "可以進行送審。"}
tool name:  check_department_budget
tool id:  call_vlCS5Gmesl8t24YQPDW7Xxpy

--------------------------------------------------

AIMessage
{"document_id":"B2048","department_id":"dept-01","total_amount":80000,"available_budget":100000,"status":"ready","check_results":"可用預算足以支付採購總金額。","recommendations":"可以進行送審。"}
tool call:  []

--------------------------------------------------

Structured Response:
Fina

## 建立協調代理人 Coordination Agent 

In [39]:
# The system prompt 
COORD_AGENT_SYSTEM_PROMPT = """
你是 Coordination Agent，負責協調 Procurement Agent、Material Agent 與 Finance Agent，判斷採購單是否具備送審條件。

你的工作流程如下：

1. 取得使用者指定的採購單編號。
2. 呼叫 Procurement Agent 檢查採購文件。
3. 呼叫 Material Agent 檢查物料需求。
4. 只有 Procurement Agent 回傳 `ready`，並確認申請部門與採購總金額後，才能呼叫 Finance Agent。
5. 整合三個 Agent 的檢查結果。

判斷規則：

- 三項檢查都是 `ready`：整體狀態為 `ready`。
- 任一項檢查為 `blocked`：整體狀態為 `blocked`。
- 沒有 `blocked`，但任一項為 `unknown`：整體狀態為 `unknown`。
- 如果採購文件檢查未通過或無法確認，不得執行財務檢查，並說明財務檢查尚未進行。

請遵守以下限制：

- 只根據各 Agent 回傳的結果判斷，不得自行重新檢查商業規則。
- 不得自行補造採購、物料或預算資料。
- 不得直接呼叫底層的採購、物料或預算檢查 Tools。
- 專業 Agent 回傳的「可以送審」只代表該項檢查通過，不代表整體已具備送審條件。
- 不得建立、修改、核准或實際送出採購單。

最終回覆應包含：

- 採購單編號
- 採購文件檢查結果
- 物料需求檢查結果
- 財務預算檢查結果
- 整體狀態：`ready`、`blocked` 或 `unknown`
- 阻擋原因與建議處理方式
- 明確說明本次只完成條件檢查，尚未實際送審
"""

In [40]:
# Wrap the three agents into tools

from langchain.tools import tool

@tool
def call_procurement_agent(document_id: str) -> dict[str, str]:
    """
    Call the Procurement Agent to check the procurement document.
    """
    return procurement_agent.invoke(
        input={"messages": [
            {"role": "user", "content": f"請檢查採購單 {document_id} 是否符合送審前的規則。"}
        ]}
    )   


In [41]:
@tool
def call_material_agent(document_id: str) -> dict[str, str]:
    """
    Call the Material Agent to check the material requirements.
    """
    return material_agent.invoke(
        input={"messages": [
            {"role": "user", "content": f"請檢查採購單 {document_id} 是否符合物料需求。"}
        ]}
    )

In [42]:
@tool
def call_finance_agent(document_id: str, total_amount: float, department_id: str) -> dict[str, str | float]:
    """
    Call the Finance Agent to check the department budget.
    """
    return finance_agent.invoke(
        input={"messages": [
            {"role": "user", "content": f"請檢查採購單 {document_id} 的申請部門編號 {department_id} 是否有足夠預算支付採購總金額 {total_amount} 元。"}
        ]}
    )  

In [43]:
# Create the agent

from langchain.agents import create_agent
coordination_agent = create_agent(
    model=MODEL_NAME,
    system_prompt=COORD_AGENT_SYSTEM_PROMPT,
    tools=[call_procurement_agent, call_material_agent, call_finance_agent]
)

In [44]:
# Test the Procurement Agent with a sample procurement document ID
test_document_id = "B2048"
result_of_coordination = coordination_agent.invoke(
    input={"messages": [
        {"role": "user", "content": f"請協助檢查採購單 {test_document_id} 是否具備送審條件。"}
    ]}
)  

In [49]:
# print the result
from lib.print_helper import print_response
print_response(result_of_coordination)

HumanMessage
請協助檢查採購單 B2048 是否具備送審條件。

--------------------------------------------------

AIMessage
tool call:  [{'name': 'call_procurement_agent', 'args': {'document_id': 'B2048'}, 'id': 'call_yLUfqz2TvuCoU6Gc3DMvQ7MS', 'type': 'tool_call'}, {'name': 'call_material_agent', 'args': {'document_id': 'B2048'}, 'id': 'call_OWPd8CbDSF8GHVpaFdsQWWVf', 'type': 'tool_call'}]

--------------------------------------------------

ToolMessage
{'messages': [HumanMessage(content='請檢查採購單 B2048 是否符合送審前的規則。', additional_kwargs={}, response_metadata={}, id='17fffa68-1c1a-43e7-9c2a-adba066d3617'), AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 499, 'total_tokens': 655, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}

In [50]:
print(result_of_coordination["messages"][-1].content)

採購單編號：B2048

1) 採購文件檢查結果
- 狀態：ready
- 部門：製造部（dept-01）
- 採購總金額：80000
- 檢查結論：採購單符合送審前的檢查規則。
- 建議：可以進行送審。

2) 物料需求檢查結果
- 採購單 document_id：B2048
- 物料_id：T1001
- 狀態：ready
- 檢查結論：採購單符合物料需求。
- 建議：可以進行送審。

3) 財務預算檢查結果
- 採購單 document_id：B2048
- 部門：dept-01
- 總金額：80000
- 可用預算：100000
- 狀態：ready
- 檢查結論：可用預算足以支付採購總金額。
- 建議：可以進行送審。

4) 整體狀態
- ready

5) 阻擋原因與建議處理方式
- 原因：無阻擋項目，三方檢查均為 ready。
- 建議：如需送審，請依流程正式提交審核即可。

6) 備註
- 本次僅完成條件檢查（三方核檢），尚未進行實際送審流程。若要正式送審，需由相關主管或審核流程執行送審動作。
